# Banking Intent Classification with Unsloth & QLoRA

This notebook fine-tunes a Llama-3 8B model for intent classification on the BANKING77 dataset.

**Requirements:**
- Kaggle GPU T4 x2 (Settings > Accelerator > GPU T4 x2)
- Internet enabled (Settings > Internet > On)
- Persistence: Files only

## Step 1: Install Dependencies & Clone Repository

In [ ]:
!pip install unsloth

In [ ]:
!git clone https://github.com/tunah72/banking-intent-unsloth.git
%cd banking-intent-unsloth
!pip install -r requirements.txt

## Step 2: Data Preprocessing

Downloads the BANKING77 dataset from HuggingFace and applies Stratified Sampling:
- Train: 50 samples/label (3,850 total)
- Validation: 5 samples/label (385 total)
- Test: 10 samples/label (770 total)

In [ ]:
!python scripts/preprocess_data.py

## Step 3: Model Fine-tuning

Fine-tunes `unsloth/llama-3-8b-bnb-4bit` using QLoRA with the following settings:
- LoRA rank: 16, alpha: 16
- Effective batch size: 8 (batch_size=2 x gradient_accumulation=4)
- Learning rate: 2e-4
- Max sequence length: 128
- Saves the best checkpoint based on validation loss

In [ ]:
!bash train.sh

## Step 4: Model Evaluation

Evaluates the fine-tuned model on the test set (770 samples) and generates:
- `results/test_predictions.csv` - Full prediction log
- `results/classification_report.txt` - Precision, Recall, F1 per label
- `results/metrics.json` - Raw metrics in JSON format
- `results/confusion_matrix.png` - Confusion matrix heatmap

In [ ]:
!bash evaluate.sh

### View Evaluation Results

In [ ]:
with open("results/classification_report.txt", "r") as f:
    print(f.read())

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/confusion_matrix.png", width=800))

## Step 5: Inference Demo

Loads the trained model and predicts intent labels for sample banking queries.

In [ ]:
!bash inference.sh

## Step 6: Save Results

Copies all outputs to `/kaggle/working/output/` for persistent storage.

After running this cell, click **Save Version** > **Save & Run All (Commit)** to store permanently.

In [ ]:
import shutil
import os

output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

# Copy model checkpoint
if os.path.exists("checkpoints/final_best_model"):
    shutil.copytree("checkpoints/final_best_model", f"{output_dir}/final_best_model", dirs_exist_ok=True)
    print("Copied model checkpoint.")

# Copy evaluation results
if os.path.exists("results"):
    shutil.copytree("results", f"{output_dir}/results", dirs_exist_ok=True)
    print("Copied evaluation results.")

# Copy preprocessed data
if os.path.exists("sample_data"):
    shutil.copytree("sample_data", f"{output_dir}/sample_data", dirs_exist_ok=True)
    print("Copied preprocessed data.")

print(f"\nAll outputs saved to {output_dir}")
print("Click Save Version to persist these files on Kaggle.")